In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

# Open by name (or use open_by_url / open_by_key if you have the sheet's URL/ID)
sheet = gc.open('Capstone Copy of Appointments').worksheet('Appointments_2025/2026')
tenure_sheet= gc.open('Capstone Copy of Appointments').worksheet('Clinician_Tenure')

# Pull all data into a DataFrame
import pandas as pd
import json
data = sheet.get_all_records()
raw_df = pd.DataFrame(data)

tenure = tenure_sheet.get_all_records()
tenure_df = pd.DataFrame(tenure)


In [ ]:
# Mount Drive first — everything below depends on this
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
Github_Token = userdata.get('Github_Token')

# Move into the repo (already cloned, lives permanently in Drive)
%cd /content/drive/MyDrive/Data_Analytics_Capstone

# Git identity (session-only, still needs to be set each time)
!git config user.name "Sharion2023"
!git config user.email "your-email@example.com"

# Activate nbstripout locally each session
!pip install nbstripout --quiet
!nbstripout --install

# Load staff_anon.json from Drive
import json
with open('/content/drive/MyDrive/staff_anon.json', 'r') as f:
    staff_anon = json.load(f)

print("✅ Drive mounted, repo location set, git configured, nbstripout active, staff_anon loaded.")

In [ ]:
# Strip whitespace on the join key in both DataFrames
raw_df['staff_member_name'] = raw_df['staff_member_name'].str.strip()
tenure_df['staff_member_name'] = tenure_df['staff_member_name'].str.strip()

# Merge
df = raw_df.merge(tenure_df, on='staff_member_name', how='left')

print(f"raw_df rows: {len(raw_df)} | merged df rows: {len(df)}")

In [ ]:
unmatched_count = df[df['start_date'].isna()]['staff_member_name'].nunique()
print(f"{unmatched_count} unique staff members did not match")

In [ ]:
!pwd

In [ ]:
!git status

In [ ]:
df_clean = df.copy()

In [ ]:
col_to_keep = [
    'patient_number',    # this is your patient ID field, not 'patient_id'
    'staff_member_name',
    'start_at',
    'arrived_at',
    'first_visit',
    'treatment_name',
    'state',             # appointment status (e.g. 'arrived') - filter out no-shows/cancellations
    'no_show_at',        # confirms whether a no-show occurred
    'cancelled_at',      # confirms whether/when cancelled
    'booked_at',         # lead time analysis for intake-conversion stream
]

df_clean = df_clean[col_to_keep]

In [ ]:
df_clean.head()


In [ ]:
df_clean['staff_member_name'].unique()


In [ ]:
df_clean['staff_member_name'] = df_clean['staff_member_name'].replace(staff_anon)

In [ ]:
df_clean['staff_member_name'].unique()

In [ ]:
df_clean.head(20)

In [ ]:
# Unique patient count
print(df_clean['patient_number'].nunique())

In [ ]:
df_clean['staff_member_name'] = df_clean['staff_member_name'].str.strip()

In [ ]:
# Visits per patient
visits_per_patient = df_clean.groupby('patient_number').size()

In [ ]:
# Class balance: did they return for a 2nd visit?
returned = (visits_per_patient >= 2).sum()
did_not_return = (visits_per_patient == 1).sum()

print(f"Returned: {returned} ({returned/(returned+did_not_return):.1%})")
print(f"Did not return: {did_not_return} ({did_not_return/(returned+did_not_return):.1%})")

In [ ]:
df_clean[df_clean['staff_member_name'] == 'Clinician D'].head(20)

In [ ]:
# Visits per patient, per clinician
visits_per_patient_by_clinician = df_clean.groupby(['staff_member_name', 'patient_number']).size()

# For each clinician, what % of their patients returned (2+ visits)?
def retention_rate(group):
    returned = (group >= 2).sum()
    total = len(group)
    return returned / total

retention_by_clinician = visits_per_patient_by_clinician.groupby('staff_member_name').apply(retention_rate)
print(retention_by_clinician)

In [ ]:
import matplotlib.pyplot as plt

# Sort for readability
sorted_retention = retention_by_clinician.sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
sorted_retention.plot(kind='barh', ax=ax, color='#4C72B0')

# Add clinic-wide average line
clinic_avg = retention_by_clinician.mean()
ax.axvline(clinic_avg, color='red', linestyle='--', linewidth=1, label=f'Clinic avg: {clinic_avg:.2f}')

ax.set_xlabel('Retention Rate (% patients with 2+ visits)')
ax.set_title('Patient Retention Rate by Clinician')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
#Get patient count per practitioner to check if this wide range is reasonable
patient_count_by_clinician = visits_per_patient_by_clinician.groupby('staff_member_name').size()
print(patient_count_by_clinician)

In [ ]:
summary = pd.DataFrame({
    'patient_count': visits_per_patient_by_clinician.groupby('staff_member_name').size(),
    'retention_rate': visits_per_patient_by_clinician.groupby('staff_member_name').apply(retention_rate)
})
print(summary.sort_values('retention_rate', ascending=False))

The variation in patient count is vast. I'll remove outliers for a cleaner interpretation.

In [ ]:
Q1 = summary['patient_count'].quantile(0.25)
Q3 = summary['patient_count'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = summary[(summary['patient_count'] < lower_bound) | (summary['patient_count'] > upper_bound)]
print(f"Bounds: {lower_bound:.0f} to {upper_bound:.0f}")
print(outliers)

IQR method not effective on such a small data set. Will attempt

In [ ]:
MIN_CASELOAD = 50  # justify this number in your methodology write-up

reliable_summary = summary[summary['patient_count'] >= MIN_CASELOAD]
excluded = summary[summary['patient_count'] < MIN_CASELOAD]

print(f"Included: {len(reliable_summary)} clinicians")
print(f"Excluded (caseload < {MIN_CASELOAD}): {len(excluded)} clinicians")
print(excluded)

In [ ]:
from scipy import stats

# --- Full dataset (all 16 clinicians) ---
corr_full, p_full = stats.pearsonr(summary['patient_count'], summary['retention_rate'])
print("=== Full dataset (all clinicians) ===")
print(f"n = {len(summary)}")
print(f"Pearson r = {corr_full:.3f}, p = {p_full:.3f}")

print()

# --- Thresholded dataset (caseload >= MIN_CASELOAD) ---
corr_thresh, p_thresh = stats.pearsonr(reliable_summary['patient_count'], reliable_summary['retention_rate'])
print(f"=== Thresholded dataset (caseload >= {MIN_CASELOAD}) ===")
print(f"n = {len(reliable_summary)}")
print(f"Pearson r = {corr_thresh:.3f}, p = {p_thresh:.3f}")